# Lab 12 · The agent graph

**Day 4 · S23** · Budget: 60 min of the 70 min slot · Runs on: Colab or a laptop, CPU only · One API key, or a saved run

**Follows** S22, which left two MCP servers running on your machine and a model calling them one hop at a time.
**Hands off to** S24, which puts the controls on whatever you decide to build here.

S22 closed on the question this lab exists to settle:

> The work has four steps, the third one depends on what the second found, and two of them could run at the same time. Who decides the order — you, in code, or the model, at runtime?

S20 called that the line between rung 4 and rung 5, and said the honest answer is usually rung 4. This lab does not repeat the claim. It builds four architectures over the same six tickets, with the same tools, the same rules and the same model, and puts the numbers in one table. The result is not the one the framework documentation prepares you for.

### The job

Six tickets off the SGP service desk queue. For each: read it, work out what it needs, and write the update you would propose — the route, the answer, the documents it rests on, the status change. Nothing is written back. Every arm stops at a proposal, the way S22's gate made it stop.

| Ticket | What it is | What it is here to test |
|---|---|---|
| SD-2026-0409 | historian logging HX-4471, caller wants to restart the service | the ordinary case. One search settles it, and the document says do not restart |
| SD-2026-0427 | HS-01 rejecting new tags, code transcribed off a phone photo as "HX 4417 or 4471" | two codes, one symptom, and a sibling ticket on the same system |
| SD-2026-0421 | the maximum discharge pressure for P-301 | there is no P-301 in the corpus. The right answer is to say so |
| SD-2026-0423 | finance want the K-301 overhaul budget | not the desk's question, and not a document's either |
| SD-2026-0435 | vendor wants a firewall rule for remote access | the flow is in one section of the standard, the approval it needs is in another |
| SD-2026-0412 | gas detector failed calibration, what has to happen before it returns to service | half the answer is in a second document that the first one only names |

The last two are the lab. The other four are there so the table has a baseline.

### The four arms

| Arm | Built in | Rung | Who decides the next step | Steps per ticket |
|---|---|---|---|---|
| **workflow** | §4 | 4 | you, in code | fixed, known before it runs once |
| **agent** | §5 | 5 | the model, at runtime | unknown until it stops |
| **two-hop** | §9 | 4, plus one more `if` | you, in code, including when to look again | fixed, known |
| **hybrid** | §10 | 4 with one 5 inside it | you, except in the one node you could not draw | fixed, plus one capped loop |

Exactly one thing differs between the first two: who chooses what happens next. Same six tickets, same two MCP servers, same model, same house rules, same output record. Whatever the table in §6 shows, that is what it is measuring. Sections 9 and 10 are built afterwards, in response to what it shows — which is the order this work happens in when it is done honestly.

### What this lab needs

| From | What | If you do not have it |
|---|---|---|
| S22 | the two MCP servers under `services/mcp_servers/` | they are in the repo. Nothing to start by hand — the client starts them |
| Lab 07 | `artifacts/rag_index/chunks.jsonl`, sitting behind the docs server | run 07 to the end, or pull the folder from the repo |
| anywhere | one OpenAI key, in `.env`, the environment, or Colab secrets | the notebook replays a saved run and every beat still lands |

## 1. Setup

Four cells: find the lab folder, get a model, describe the two servers, see what they offer.

In [1]:
# Setup: find the lab folder, detect the runtime, install pinned packages on Colab.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = os.environ.get("LAB_REPO_URL", "")  # Colab: the course repo URL, once it is published


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "services" / "mcp_servers" / "sgp_docs.py").exists():
            return p


ROOT = find_root()
if ROOT is None and IN_COLAB and REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/lab"], check=True)
    ROOT = Path("/content/lab")
if ROOT is None:
    raise RuntimeError("Lab folder not found. Open this notebook from inside it, or set LAB_REPO_URL on Colab.")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.2.0", "openai==3.0.0",
                    "python-dotenv==1.1.0", "rank-bm25==0.2.2", "tabulate==0.9.0"], check=True)
sys.path.insert(0, str(ROOT / "scripts"))
print("lab folder:", ROOT, "| runtime:", "Colab" if IN_COLAB else "local")

lab folder: /Users/drpreetyrai./aiguru | runtime: local


The model, and a counter wrapped around it.

Every model call in this notebook — both arms, every node — goes through the same `Meter`. That is not tidiness. A comparison whose cost column is estimated is a comparison somebody will argue with, and the argument will be about the estimate rather than about the architecture. It is also the seam S24 turns into a budget: you cannot cap what you do not count.

In [2]:
import asyncio
import json
import re
import time
from dataclasses import dataclass, field

import pandas as pd
from vision_client import load_openai_key, openai_client

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 160)

LAB = "12_agent_graph"
OUT = ROOT / "outputs" / LAB
(OUT / "traces").mkdir(parents=True, exist_ok=True)
PREBAKED = Path(os.environ.get("LAB_PREBAKED_DIR", ROOT / "facilitator" / "prebaked_outputs")) / LAB
MODEL = os.environ.get("LAB_MODEL", "gpt-4.1-mini")
FORCE = False  # True re-runs every arm instead of reading outputs/12_agent_graph/runs/
# USD per million tokens, input/output, list price at the time of writing. Edit to your own
# contract: the cost column below is the one your finance business partner will ask about.
PRICE = {"gpt-4.1-mini": (0.40, 1.60)}


class Meter:
    """The model client with a counter around it. Both arms are billed through one object."""

    def __init__(self, client):
        self.client = client
        self.reset()

    def reset(self):
        self.calls = self.prompt_tokens = self.completion_tokens = 0
        self.seconds = 0.0

    def take(self) -> dict:
        """Read the counters and zero them, so the next thing measured starts from nothing."""
        spent = {"model_calls": self.calls, "prompt_tokens": self.prompt_tokens,
                 "completion_tokens": self.completion_tokens, "model_seconds": round(self.seconds, 2)}
        self.reset()
        return spent

    @property
    def responses(self):  # so this stands in for the OpenAI client wherever one is expected
        return self

    def create(self, **kwargs):
        start = time.time()
        response = self.client.responses.create(**kwargs)
        self.seconds += time.time() - start
        self.calls += 1
        usage = getattr(response, "usage", None)
        if usage is not None:
            self.prompt_tokens += usage.input_tokens
            self.completion_tokens += usage.output_tokens
        return response


def usd(prompt_tokens: float, completion_tokens: float, model: str = MODEL) -> float:
    rate_in, rate_out = PRICE.get(model, (0.0, 0.0))
    return round((prompt_tokens * rate_in + completion_tokens * rate_out) / 1e6, 5)


OPENAI = None
if load_openai_key(ROOT):
    try:
        OPENAI = openai_client()
        OPENAI.responses.create(model=MODEL, input=[{"role": "user", "content": "reply with: ok"}])
    except Exception as e:  # a key can be present and still not work; find out here, not mid-lab
        print(f"model unreachable: {type(e).__name__}: {str(e)[:160]}")
        OPENAI = None
HAVE_MODEL = OPENAI is not None
METER = Meter(OPENAI)
print("model:", f"{MODEL}, reachable" if HAVE_MODEL
      else "unavailable — the arms replay from outputs/ or facilitator/prebaked_outputs/")

model: gpt-4.1-mini, reachable


The two servers from S22, unchanged. You do not start them: the client starts them, on stdio, when it connects — which is the part of MCP that stops feeling like a protocol and starts feeling like a subprocess.

Two environment settings are worth reading rather than skipping, because both are authority decisions made outside the model:

- `SGP_DOCS_RETRIEVAL=bm25` — the fast lexical retriever, so nobody waits for an embedder. Switch it to `hybrid` in section 12 and lab 07's full pipeline is behind the same tool contract, with nothing else changing.
- `SGP_DESK_STORE` points at a ticket file inside `outputs/`. Disposable by design. Delete it and the next call re-seeds from the twelve tickets everybody started with.

In [3]:
from mcp import StdioServerParameters
from mcp_bridge import McpTools, allow_all, propose_only, run_agent

PY = sys.executable
DOCS = StdioServerParameters(
    command=PY,
    args=[str(ROOT / "services" / "mcp_servers" / "sgp_docs.py")],
    env={**os.environ, "SGP_DOCS_RETRIEVAL": os.environ.get("SGP_DOCS_RETRIEVAL", "bm25")},
)
DESK = StdioServerParameters(
    command=PY,
    args=[str(ROOT / "services" / "mcp_servers" / "sgp_servicedesk.py")],
    env={**os.environ, "SGP_DESK_STORE": str(OUT / "tickets.json"), "SGP_DESK_ACTOR": "lab12-client"},
)

async with McpTools({"docs": DOCS, "desk": DESK}) as probe:
    TOOLS_OFFERED = pd.DataFrame(
        [{"tool": name, "server": spec["server"], "read_only": spec["read_only"],
          "description": spec["description"].split(".")[0]} for name, spec in probe.tools.items()]
    )
TOOLS_OFFERED

,tool,server,read_only,description
0,docs__search_documents,docs,True,Search the Sabkha Gas Plant document set and return the passages t...
1,docs__get_document,docs,True,"Read one document in full, by id"
2,docs__list_documents,docs,True,"Every document id in the corpus, with its revisions"
3,desk__list_tickets,desk,True,"The live ticket queue, one summary row per ticket"
4,desk__get_ticket,desk,True,"One ticket in full: the original text, the live status, the assign..."
5,desk__find_similar_tickets,desk,True,"Past tickets that read like this one, most alike first, with how t..."
6,desk__update_ticket,desk,False,"Change a ticket on the live service desk: set its status, reassign..."


Seven tools, one of which writes. Every arm below is handed exactly this list, and every arm is gated with `propose_only` from S22 — reads run, writes come back as a refusal the model can read.

That is deliberate. This lab changes one variable, and it is not authority. S24 changes that one.

## 2. The job, and how it is scored

Three checks, all mechanical. No model judges another model here: a judge that drifts makes the comparison unarguable in the wrong direction, and these six answers are short enough to check with a regex.

| Check | Passes when |
|---|---|
| `route` | the arm's route is one the desk would accept |
| `cited` | every document the answer rests on is cited. Scored only on the four tickets that need one |
| `complete` | the facts the answer must carry are in it, and the ones it must not invent are not |

`cited` is the check the last two tickets were chosen for.

**SD-2026-0435.** The permitted remote-support flow is in MAN-FW-01 section 3. That a firewall change needs an approved management of change and the change advisory board is in section 2. One search brings back the section that matches the words in the ticket, which is section 3.

**SD-2026-0412.** MAN-GD-01 says a detector that fails calibration is inhibited under an override permit and gets a new sensor head. What an override permit requires — Area Authority approval, compensating measures, the override register, 72 hours before it becomes a management of change — is in MAN-SIS-01, which MAN-GD-01 names and does not contain.

In both cases one search returns the first half of the answer, and nothing in what comes back tells you the second half is missing.

In [4]:
# The six tickets, and the answer every arm is held to. Regexes, so the scoring is arguable
# in public: if you disagree with a row, change the row and re-run — that is what a gold set is for.
GOLD = [
    {"ticket_id": "SD-2026-0409", "one_hop": True,
     "routes": ["documents", "escalate"], "cite": ["MAN-HIS-01"],
     "must": [r"HX-?4471", r"(?i)discards? the write queue|(do not|never|not)\s+restart"], "must_not": [],
     "about": "archive write queue overflow, a restart discards the queue. Two routes are "
              "defensible here: the document answers it, and the document also says raise a P2"},
    {"ticket_id": "SD-2026-0427", "one_hop": True,
     "routes": ["documents"], "cite": ["MAN-HIS-01"],
     "must": [r"HX-?4417", r"(?i)licen[cs]e|tag count"], "must_not": [],
     "about": "new tags rejected, existing still collecting: the licence code, not the archive one"},
    {"ticket_id": "SD-2026-0421", "one_hop": True,
     "routes": ["no_document"], "cite": [],
     "must": [], "must_not": [r"\d+\s*barg", r"MAN-P-20[12]"],
     "about": "there is no P-301 in the corpus; P-201's 64 barg belongs to another pump"},
    {"ticket_id": "SD-2026-0423", "one_hop": True,
     "routes": ["route_elsewhere", "no_document"], "cite": [],
     "must": [], "must_not": [r"(?i)\bOMR\b|\bUSD\b|\$\s?\d"],
     "about": "a budget is not in the document store and not the desk's to give"},
    {"ticket_id": "SD-2026-0435", "one_hop": False,
     "routes": ["documents"], "cite": ["MAN-FW-01"],
     "must": [r"(?i)jump host", r"(?i)\bCAB\b|change advisory|management of change|MOC|HSE-PRO-060"],
     "must_not": [],
     "about": "the flow is section 3, the approval it needs is section 2"},
    {"ticket_id": "SD-2026-0412", "one_hop": False,
     "routes": ["documents"], "cite": ["MAN-GD-01", "MAN-SIS-01"],
     "must": [r"(?i)bump test|calibration", r"(?i)override (permit|register)"], "must_not": [],
     "about": "MAN-GD-01 names the override permit; MAN-SIS-01 is what it means"},
]
TICKET_IDS = [g["ticket_id"] for g in GOLD]
GOLD_BY_ID = {g["ticket_id"]: g for g in GOLD}


def score(record: dict, gold: dict) -> dict:
    """Three booleans against one proposed record. Text checks read the answer only —
    what an arm admits it could not reach is measured separately, and it is worth more."""
    answer = record.get("answer", "") or ""
    cites = " ".join(record.get("citations", []) or [])
    return {
        "route": record.get("route") in gold["routes"],
        # not applicable where no document should be cited: two of the six tickets are like that,
        # and scoring them as passes would flatter every arm equally and tell you nothing
        "cited": float(all(doc in cites for doc in gold["cite"])) if gold["cite"] else float("nan"),
        "complete": (all(re.search(p, answer) for p in gold["must"])
                     and not any(re.search(p, answer + " " + cites) for p in gold["must_not"])),
    }


pd.DataFrame([{"ticket": g["ticket_id"], "one search is enough": g["one_hop"],
               "route": "/".join(g["routes"]), "must cite": ", ".join(g["cite"]) or "nothing",
               "why it is in the set": g["about"]} for g in GOLD])

,ticket,one search is enough,route,must cite,why it is in the set
0,SD-2026-0409,True,documents/escalate,MAN-HIS-01,"archive write queue overflow, a restart discards the queue. Two ro..."
1,SD-2026-0427,True,documents,MAN-HIS-01,"new tags rejected, existing still collecting: the licence code, no..."
2,SD-2026-0421,True,no_document,nothing,there is no P-301 in the corpus; P-201's 64 barg belongs to anothe...
3,SD-2026-0423,True,route_elsewhere/no_document,nothing,a budget is not in the document store and not the desk's to give
4,SD-2026-0435,False,documents,MAN-FW-01,"the flow is section 3, the approval it needs is section 2"
5,SD-2026-0412,False,documents,"MAN-GD-01, MAN-SIS-01",MAN-GD-01 names the override permit; MAN-SIS-01 is what it means


## 3. Draw it before you write it

S20's sentence, when it was still only a claim: *if you can draw the flowchart, build the flowchart.* Here is the flowchart for triaging a ticket, drawn before any code exists.

1. read the ticket
2. find tickets like it, and every other ticket on the same system — **neither needs the other, so they run together**
3. decide the route, and write the one search query that route needs
4. unless the ticket is not the desk's to answer at all, search the document store — once
5. write the proposal

Five layers, one branch, one parallel pair. Nothing in that needs a model to schedule it, and a model asked to schedule it will rediscover it on every ticket, slowly, at a price.

The next cell is the runner. A node has a name, the nodes it reads, and a function; nodes whose dependencies are all met run together. That is the whole framework — about forty lines — and it is written out rather than imported to make one point: **there is no magic in the box.** Every graph framework you will be shown this year is this, plus retries, plus a dashboard, plus a vendor.

In [5]:
@dataclass
class Node:
    name: str
    needs: tuple           # the nodes this one reads. These are the edges, and they are the whole graph
    fn: object             # async (state) -> value
    kind: str = "tool"     # tool | model | agent — what it costs, and who decides inside it
    note: str = ""         # what it does, for the drawing
    when: object = None    # the switch. A branch you can enumerate is an `if`, not autonomy
    when_note: str = ""


class Graph:
    """Nodes and the nodes they read. Every edge below is a line somebody wrote and can be shown
    to somebody else. That is the entire difference between rung 4 and rung 5."""

    def __init__(self, name: str, nodes: list):
        self.name, self.nodes = name, nodes

    def layers(self) -> list:
        done, out, left = set(), [], list(self.nodes)
        while left:
            layer = [n for n in left if set(n.needs) <= done]
            if not layer:
                raise ValueError(f"cycle or missing dependency: {[n.name for n in left]}")
            out.append(layer)
            done |= {n.name for n in layer}
            left = [n for n in left if n.name not in done]
        return out

    async def run(self, **state):
        state, log = dict(state), []
        for depth, layer in enumerate(self.layers(), 1):
            todo = [n for n in layer if n.when is None or n.when(state)]
            for skipped in [n for n in layer if n not in todo]:
                state[skipped.name] = None
                log.append({"layer": depth, "node": skipped.name, "kind": skipped.kind,
                            "ran": False, "seconds": 0.0})

            async def timed(node):
                start = time.time()
                return node, await node.fn(state), round(time.time() - start, 2)

            for node, value, seconds in await asyncio.gather(*(timed(n) for n in todo)):
                state[node.name] = value
                log.append({"layer": depth, "node": node.name, "kind": node.kind,
                            "ran": True, "seconds": seconds})
        return state, log

    def draw(self) -> str:
        counts = {k: sum(n.kind == k for n in self.nodes) for k in ("tool", "model", "agent")}
        head = (f"{self.name}: {len(self.nodes)} nodes in {len(self.layers())} layers, "
                f"{counts['model']} model calls, up to {counts['tool']} tool calls"
                + (f", {counts['agent']} capped loop" if counts["agent"] else ""))
        lines = [head, ""]
        for depth, layer in enumerate(self.layers(), 1):
            for i, n in enumerate(layer):
                together = "  <- this layer runs together" if len(layer) > 1 and i == 0 else ""
                gate = f"  [{n.when_note}]" if n.when_note else ""
                lines.append(f"  {depth}  {n.name:<12} {n.kind:<6} {n.note}{gate}{together}")
        return "\n".join(lines)

    def mermaid(self) -> str:
        shape = {"tool": '{0}["{0}"]', "model": '{0}("{0}")', "agent": '{0}{{{{"{0}"}}}}'}
        out = ["flowchart TD"] + [f"    {shape[n.kind].format(n.name)}" for n in self.nodes]
        out += [f"    {dep} --> {n.name}" for n in self.nodes for dep in n.needs]
        return "\n".join(out)


print("Graph, Node ready")

Graph, Node ready


Two helpers the nodes need, and one convention worth naming.

`call()` makes a tool call **because the code said so**. `ask_json()` makes a model call that must come back in a shape you declared — rung 1 from S20's table, doing its job inside a node. Both record what they did, and `chosen_by` on every logged call is the column this lab is about: `code` or `model`.

`ask_json` is blocking, so nodes await it on a thread. That is what lets layer 2 genuinely run two things at once instead of pretending to.

In [6]:
def as_json(text: str):
    """MCP sends one text block per returned item, so a list tool arrives as several JSON objects
    in a row. Decode them all; hand back one, or a list."""
    out, index, decoder = [], 0, json.JSONDecoder()
    try:
        while index < len(text):
            while index < len(text) and text[index].isspace():
                index += 1
            if index >= len(text):
                break
            obj, index = decoder.raw_decode(text, index)
            out.append(obj)
    except ValueError:
        return {"error": text[:400]}
    return out[0] if len(out) == 1 else out


async def call(state: dict, name: str, args: dict):
    """One tool call the graph makes because the code said so."""
    start = time.time()
    text = await state["tools"].call(name, args)
    state["calls"].append({"tool": name, "args": args, "chosen_by": "code",
                           "seconds": round(time.time() - start, 2)})
    return as_json(text)


def ask_json(prompt: str, schema: dict, name: str = "record") -> dict:
    """A constrained call: the answer arrives in a shape your code can use without a regex."""
    response = METER.create(model=MODEL, input=[{"role": "user", "content": prompt}],
                            text={"format": {"type": "json_schema", "name": name,
                                             "schema": schema, "strict": True}},
                            temperature=0)
    return json.loads(response.output_text)


async def ask_json_async(prompt: str, schema: dict, name: str = "record") -> dict:
    return await asyncio.to_thread(ask_json, prompt, schema, name)


def brief(ticket: dict) -> str:
    keep = ("ticket_id", "status", "priority", "category", "affected_system", "summary",
            "assignee", "sla_breached", "hours_to_sla", "latest_note", "allowed_next_status")
    return json.dumps({k: ticket.get(k) for k in keep}, indent=1)


def passages(hits, limit: int = 5) -> str:
    hits = hits if isinstance(hits, list) else [hits]
    return "\n\n".join(
        f"[{h.get('doc_id')} rev {h.get('revision')}, {h.get('status')}] {h.get('section')}\n{h.get('text', '')[:700]}"
        for h in hits[:limit] if isinstance(h, dict) and h.get("doc_id")
    ) or "(nothing came back)"


print("call, ask_json, brief, passages ready")

call, ask_json, brief, passages ready


The house rules, written once. Both arms get exactly these, because a comparison where one side has a better prompt measures the prompt.

In [7]:
HOUSE_RULES = """You are triaging tickets on the Sabkha Gas Plant IT service desk.

Routes, pick exactly one:
  duplicate        another service desk ticket is already working the same fault, and you can name
                   it. A different fault on the same system is not a duplicate, a work order is not
                   a ticket, and the similarity search scoring a ticket's own text highly is not a
                   duplicate either
  documents        a plant document settles it: a setpoint, an error code, a procedure step
  escalate         it needs a person now: safety exposure, a P1, an SLA breached with no owner
  route_elsewhere  it is not the IT service desk's to answer (finance, procurement, HR, budgets)
  no_document      it needs a document and the document store does not cover it

Rules:
1. Cite the document id and revision for every number, code and procedure step you state.
   Cite only documents you have actually read in this session.
2. If the documents do not name the system or the code the ticket is about, the route is
   no_document, citations are empty, and you say so. Retrieval always returns something;
   something is not the same as coverage.
3. An answer is complete only when every step it depends on comes from a document you have read.
   If what you read names another document or another section for part of the answer, read that
   one too when you are able to. When you are not able to, say exactly what is missing.
4. Propose a status only from the ticket's allowed_next_status. Propose; never write."""

# The one sentence that decides whether an answer stops early. Every arm is given it, word for
# word: the workflow in its two prompts, the agent in its task, the hybrid in its agent node.
# Only one of the three is able to act on it, and that is the finding, not the wording.
COMPLETENESS = ("An answer is complete only when every step it depends on comes from a document you "
                "have read. If a document you read names another document, or another section, for "
                "part of the answer, read that one too.")

PROPOSAL_SCHEMA = {
    "type": "object",
    "properties": {
        "route": {"type": "string",
                  "enum": ["duplicate", "documents", "escalate", "route_elsewhere", "no_document"]},
        "answer": {"type": "string", "description": "What the desk writes back to the caller, with citations inline."},
        "citations": {"type": "array", "items": {"type": "string"},
                      "description": "Document ids with revisions, e.g. 'MAN-HIS-01 rev 5'."},
        "related_tickets": {"type": "array", "items": {"type": "string"}},
        "proposed_status": {"type": "string"},
        "proposed_assignee": {"type": "string"},
        "missing": {"type": "string",
                    "description": "What you could not reach and would need next. Empty if nothing."},
    },
    "required": ["route", "answer", "citations", "related_tickets",
                 "proposed_status", "proposed_assignee", "missing"],
    "additionalProperties": False,
}
print("house rules:", len(HOUSE_RULES.split()), "words, shared by every arm")

house rules: 253 words, shared by every arm


Now the five nodes. Read the `needs` tuples rather than the function bodies: those are the edges, and the edges are the architecture.

In [8]:
TRIAGE_SCHEMA = {
    "type": "object",
    "properties": {
        "route": {"type": "string",
                  "enum": ["duplicate", "documents", "escalate", "route_elsewhere", "no_document"]},
        "query": {"type": "string", "description": "The one search query. You do not get a second."},
        "duplicate_of": {"type": "string",
                         "description": "For route=duplicate only: the ticket id it duplicates. Empty otherwise."},
        "reason": {"type": "string"},
    },
    "required": ["route", "query", "duplicate_of", "reason"],
    "additionalProperties": False,
}


async def n_ticket(s):
    return await call(s, "desk__get_ticket", {"ticket_id": s["ticket_id"]})


async def n_similar(s):
    hits = await call(s, "desk__find_similar_tickets", {"text": s["ticket"]["summary"], "k": 4})
    hits = hits if isinstance(hits, list) else [hits]
    # A similarity search over a store that contains the query returns the query, at the top, every
    # time. Left in, a classifier reads it as "an existing ticket with identical text" and routes
    # every ticket as a duplicate of itself. One line, and you will meet it again in week one.
    return [h for h in hits if h.get("ticket_id") != s["ticket_id"]][:3]


async def n_queue(s):
    return await call(s, "desk__list_tickets",
                      {"status": "", "affected_system": s["ticket"]["affected_system"]})


async def n_triage(s):
    prompt = f"""{HOUSE_RULES}

TICKET
{brief(s['ticket'])}

TICKETS THAT READ LIKE IT (the desk's own similarity search)
{json.dumps(s['similar'], indent=1)[:1800]}

EVERY TICKET ON {s['ticket']['affected_system']}
{json.dumps(s['queue'], indent=1)[:1800]}

{COMPLETENESS}

Pick the route. Then write the single search query you would run against the plant document
store — one query, chosen now, before you have seen any passage. You do not get a second."""
    return await ask_json_async(prompt, TRIAGE_SCHEMA, "triage")


async def n_evidence(s):
    return await call(s, "docs__search_documents", {"query": s["triage"]["query"], "k": 5})


def proposal_prompt(ticket, route_hint: str, evidence: str) -> str:
    return f"""{HOUSE_RULES}

{COMPLETENESS}

TICKET
{brief(ticket)}

ROUTE CHOSEN EARLIER: {route_hint or '(none: decide it yourself)'}

WHAT CAME BACK FROM THE DOCUMENT STORE
{evidence}

Write the update the desk would propose. Confirm or correct the route against what you actually
read. Cite only what is above. If what is above does not cover the system or the code the ticket
names, the route is no_document and citations are empty."""


async def n_proposal(s):
    evidence = passages(s["evidence"]) if s["evidence"] is not None else "(no search was run)"
    return await ask_json_async(proposal_prompt(s["ticket"], s["triage"]["route"], evidence),
                                PROPOSAL_SCHEMA, "proposal")


WORKFLOW = Graph("workflow-triage", [
    Node("ticket",   (),                               n_ticket,   "tool",  "desk__get_ticket"),
    Node("similar",  ("ticket",),                      n_similar,  "tool",  "desk__find_similar_tickets"),
    Node("queue",    ("ticket",),                      n_queue,    "tool",  "desk__list_tickets"),
    Node("triage",   ("ticket", "similar", "queue"),   n_triage,   "model", "route + the one query"),
    Node("evidence", ("triage",),                      n_evidence, "tool",  "docs__search_documents",
         when=lambda s: s["triage"]["route"] != "route_elsewhere",
         when_note="skipped when the ticket is not ours to answer"),
    Node("proposal", ("ticket", "triage", "evidence"), n_proposal, "model", "the record the desk would write"),
])

print(WORKFLOW.draw())

workflow-triage: 6 nodes in 5 layers, 2 model calls, up to 4 tool calls

  1  ticket       tool   desk__get_ticket
  2  similar      tool   desk__find_similar_tickets  <- this layer runs together
  2  queue        tool   desk__list_tickets
  3  triage       model  route + the one query
  4  evidence     tool   docs__search_documents  [skipped when the ticket is not ours to answer]
  5  proposal     model  the record the desk would write


That drawing is generated from the same object the runner executes, which is the only kind of
architecture diagram worth having: it cannot go stale, because a node that is not in the picture is not in the run.

The mermaid version goes into `outputs/12_agent_graph/graph.md`. Paste it into your design doc — the one your reviewer reads instead of the code.

In [9]:
graph_md = OUT / "graph.md"
graph_md.write_text(
    f"# {WORKFLOW.name}\n\n```\n{WORKFLOW.draw()}\n```\n\n```mermaid\n{WORKFLOW.mermaid()}\n```\n",
    encoding="utf-8")
print(WORKFLOW.mermaid())
print("\nwrote", graph_md.relative_to(ROOT))

flowchart TD
    ticket["ticket"]
    similar["similar"]
    queue["queue"]
    triage("triage")
    evidence["evidence"]
    proposal("proposal")
    ticket --> similar
    ticket --> queue
    ticket --> triage
    similar --> triage
    queue --> triage
    triage --> evidence
    ticket --> proposal
    triage --> proposal
    evidence --> proposal

wrote outputs/12_agent_graph/graph.md


## 4. Arm A: the workflow (rung 4)

One ticket first, with the node log on. Watch what stays fixed: the same five layers, the same two model calls, in the same order, whatever the ticket turns out to be.

In [10]:
AGENT_STEPS = 8  # the cap. S20: an agent without one is not a design, it is an outage waiting for a Thursday
RUNS = OUT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)


async def run_arm(arm: str, per_ticket, force: bool = False, ticket_ids=None) -> list:
    """Run one arm over the ticket set, sequentially, so every row's cost is that row's cost.
    Saved to outputs/, reloaded next time, replayed from the prebaked folder when there is no model."""
    ticket_ids = ticket_ids or TICKET_IDS
    path = RUNS / f"{arm}.json"
    if path.exists() and not (force or FORCE):
        rows = json.loads(path.read_text(encoding="utf-8"))
        print(f"{arm}: {len(rows)} runs loaded from {path.relative_to(ROOT)} (set FORCE=True to re-run)")
        return rows
    if not HAVE_MODEL:
        baked = PREBAKED / "runs" / f"{arm}.json"
        if baked.exists():
            rows = json.loads(baked.read_text(encoding="utf-8"))
            print(f"{arm}: {len(rows)} runs replayed from the prebaked folder")
            return rows
        raise RuntimeError(f"No model, and no prebaked run at {baked}. Ask the facilitator.")
    rows = []
    # Three connections: both servers for the arms that use both, and the document server on its
    # own for the hybrid's agent node. A server is a subprocess; an idle one costs nothing.
    async with McpTools({"docs": DOCS, "desk": DESK}) as tools, McpTools({"docs": DOCS}) as docs_only:
        for ticket_id in ticket_ids:
            rows.append(await per_ticket(tools, docs_only, ticket_id))
            row = rows[-1]
            print(f"  {ticket_id}  {row['steps']} steps, {len(row['calls'])} tool calls, "
                  f"{row['model_calls']} model calls, {row['seconds']}s -> {row['record']['route']}")
    path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{arm}: {len(rows)} runs written to {path.relative_to(ROOT)}")
    return rows


async def workflow_once(tools, docs_only, ticket_id: str, verbose: bool = False) -> dict:
    METER.take()  # zero the counters: what follows is this ticket's bill and nothing else
    start = time.time()
    state, log = await WORKFLOW.run(ticket_id=ticket_id, tools=tools, calls=[])
    row = {"arm": "workflow", "ticket_id": ticket_id, "record": state["proposal"],
           "triage": state["triage"], "calls": state["calls"], "log": log,
           "steps": sum(r["ran"] for r in log),
           "capped": False, "seconds": round(time.time() - start, 2), **METER.take()}
    if verbose:
        print(pd.DataFrame(log).to_string(index=False))
    return row


if HAVE_MODEL:
    async with McpTools({"docs": DOCS, "desk": DESK}) as tools, McpTools({"docs": DOCS}) as docs_only:
        demo = await workflow_once(tools, docs_only, "SD-2026-0412", verbose=True)
    print("\nroute:", demo["record"]["route"], "| citations:", demo["record"]["citations"])
    print("missing:", demo["record"]["missing"] or "(nothing reported)")
else:
    print("no model: section 6 reads the saved runs instead")

 layer     node  kind  ran  seconds
     1   ticket  tool True     0.06
     2  similar  tool True     0.00
     2    queue  tool True     0.00
     3   triage model True     1.61
     4 evidence  tool True     0.06
     5 proposal model True     3.41



route: documents | citations: ['MAN-GD-01 rev 2', 'LOG-2026-06-11-N rev 1']
missing: (nothing reported)


Five layers, six nodes, two of them run together. The `seconds` column is the honest one: over a stdio server on your own machine, running `similar` and `queue` at the same time saves a few hundredths of a second and proves nothing. Point it at ServiceNow and SAP over a corporate network, where each of those is half a second or more, and the same two lines of graph halve the step. The lesson is not the saving here; it is that **you can only fan out the calls you control**. An agent loop cannot do this at all — each of its calls waits for the model to decide there should be a next one.

Now the whole set.

In [11]:
WF = await run_arm("workflow", workflow_once)

workflow: 6 runs loaded from outputs/12_agent_graph/runs/workflow.json (set FORCE=True to re-run)


Look at the per-ticket line. Two model calls and three or four tool calls, every ticket, whatever it says. That number is not an average taken after the fact — it is a property of the drawing, and you could have put it in a capacity plan before writing a line of this.

## 5. Arm B: the agent (rung 5)

Same servers, same rules, same model, same cap on nothing else. The difference is that no line below says which tool to call. The loop hands the model every tool both servers offer and asks it for the next action until it stops asking, or until the step cap bites at eight.

The task text does not come from this notebook either. It comes from the service desk server, which ships a `triage_ticket` prompt — the third MCP primitive from S22, the one everybody forgets. The desk's house rules, versioned with the server, handed to whichever client asks for them.

That is worth a beat in the room: **the workflow does not have to live in your code.** It can live in the server, next to the tools it sequences, maintained by the team who owns the system. What it cannot do from there is enforce itself, which is the rest of this lab.

In [12]:
async with McpTools({"desk": DESK}) as probe:
    got = await probe.clients["desk"].get_prompt("triage_ticket", {"ticket_id": "SD-2026-0412"})
    SERVER_PROMPT = got.messages[0].content.text
print(SERVER_PROMPT)

Triage service desk ticket SD-2026-0412.

Work in this order:
1. get_ticket. If it is already resolved or closed, stop and say so — do not answer it again.
2. find_similar_tickets on its summary. Say whether it is a duplicate.
3. If it needs a procedure or a setpoint, search the documents server and cite the document id and revision. If the documents do not cover it, say that instead of guessing.
4. Propose the update — status, assignee, note — and stop. Do not call update_ticket until the human has said yes to that exact change.



In [13]:
def shape(answer_text: str, ticket_id: str) -> dict:
    """The last step of every arm: one constrained call that puts a free-text answer into the
    record. Rung 1 composes with every rung above it, and both arms pay for it exactly once,
    so the table below compares architectures rather than output formats."""
    prompt = (f"{HOUSE_RULES}\n\nBelow is a triage note written for ticket {ticket_id}. Put it into "
              "the record without adding anything it does not say. Cite only documents it cites; "
              "if it cites none, citations is empty.\n\nNOTE\n" + answer_text)
    return ask_json(prompt, PROPOSAL_SCHEMA, "proposal")


async def agent_once(tools, docs_only, ticket_id: str, verbose: bool = False) -> dict:
    METER.take()
    start = time.time()
    got = await tools.clients["desk"].get_prompt("triage_ticket", {"ticket_id": ticket_id})
    task = f"{got.messages[0].content.text}\n{COMPLETENESS}"
    result = await run_agent(tools, task, client=METER, model=MODEL, gate=propose_only,
                             system=HOUSE_RULES, max_steps=AGENT_STEPS, verbose=verbose)
    record = await asyncio.to_thread(shape, result["answer"], ticket_id)
    calls = [{"tool": t["tool"], "args": t["args"], "chosen_by": "model", "allowed": t["allowed"],
              "seconds": None} for t in result["trace"]]
    return {"arm": "agent", "ticket_id": ticket_id, "record": record, "calls": calls,
            "trace": result["trace"], "answer": result["answer"], "steps": result["steps"],
            "capped": result["capped"], "seconds": round(time.time() - start, 2), **METER.take()}


if HAVE_MODEL:
    async with McpTools({"docs": DOCS, "desk": DESK}) as tools, McpTools({"docs": DOCS}) as docs_only:
        demo_agent = await agent_once(tools, docs_only, "SD-2026-0412", verbose=True)
    print("\nroute:", demo_agent["record"]["route"], "| citations:", demo_agent["record"]["citations"])
else:
    print("no model: section 6 reads the saved runs instead")

  step 1 -> desk__get_ticket({"ticket_id": "SD-2026-0412"})
           {   "ticket_id": "SD-2026-0412",   "status": "in_progress",   "priority": 2,   "category": "field_instrument",   "affected_system": "GD-3107",   "summary": "Gas


  step 2 -> desk__find_similar_tickets({"text": "Gas detector GD-3107 failed its six-monthly calibration this morning, span reading about 30 % low. I)
           {   "ticket_id": "SD-2026-0412",   "status": "in_progress",   "priority": 2,   "category": "field_instrument",   "affected_system": "GD-3107",   "assignee": "in


  step 3 -> docs__search_documents({"query": "gas detector calibration bump test inhibit panel", "k": 5, "include_superseded": false})
           {   "doc_id": "MAN-GD-01",   "title": "Fixed H2S Gas Detectors GD-3101 to GD-3120 - Maintenance Manual",   "section": "3. Testing and calibration",   "revision"



route: documents | citations: ['MAN-GD-01 rev 2', 'LOG-2026-06-11-N rev 1']


Two things in that trace, before any table.

**It re-derived your flowchart — because the server handed it one, in prose.** `get_ticket`, then `find_similar_tickets`, then a document search. Now read the `triage_ticket` prompt printed above again: *work in this order: 1, 2, 3, 4.* That is a flowchart. It is section 3's flowchart, written in English instead of Python, handed to a loop that re-reads and re-interprets it on every ticket at full price. S20 said most systems sold internally as agents are workflows whose authors did not want to write the switch statement. That is one, and we wrote it ourselves without noticing.

**What the loop adds is not a different plan. It is permission to leave the plan.** Nothing in this arm forced those three calls. The model could have searched twice, read a document in full, gone back to the desk for a linked ticket. Whether it used that permission is the whole question, and the next table answers it in a way the framework documentation does not prepare you for.

Now the whole set.

In [14]:
AG = await run_arm("agent", agent_once)

agent: 6 runs loaded from outputs/12_agent_graph/runs/agent.json (set FORCE=True to re-run)


## 6. The table

Six tickets, two architectures, one scoring function. Read the two halves separately: the average over all six hides the thing worth seeing.

In [15]:
def rows_to_frame(rows: list) -> pd.DataFrame:
    out = []
    for r in rows:
        gold = GOLD_BY_ID[r["ticket_id"]]
        out.append({"arm": r["arm"], "ticket": r["ticket_id"], "one_search_enough": gold["one_hop"],
                    **score(r["record"], gold), "route_taken": r["record"]["route"],
                    "tool_calls": len(r["calls"]), "model_calls": r["model_calls"],
                    "tokens": r["prompt_tokens"] + r["completion_tokens"],
                    "usd": usd(r["prompt_tokens"], r["completion_tokens"]),
                    "seconds": r["seconds"], "capped": r["capped"],
                    "admits_missing": bool((r["record"].get("missing") or "").strip())})
    return pd.DataFrame(out)


RESULTS = pd.concat([rows_to_frame(WF), rows_to_frame(AG)], ignore_index=True)
ORDER = ["workflow", "agent"]

headline = RESULTS.groupby("arm").agg(
    route=("route", "mean"), cited=("cited", "mean"), complete=("complete", "mean"),
    tool_calls=("tool_calls", "mean"), model_calls=("model_calls", "mean"),
    tokens=("tokens", "mean"), usd_per_1000=("usd", lambda c: c.mean() * 1000),
    seconds=("seconds", "median"),
).reindex(ORDER).round(2)
headline

,route,cited,complete,tool_calls,model_calls,tokens,usd_per_1000,seconds
arm,,,,,,,,
workflow,0.83,0.75,0.83,3.83,2.00,3055.83,1.60,5.74
agent,1.00,0.75,0.83,2.83,4.83,10809.17,5.08,14.99


Before reading it: `usd_per_1000` is what a thousand tickets cost, because per ticket the number rounds to nothing and nobody budgets in units that round to nothing. `seconds` is a median — one stalled vendor call would otherwise decide the column, and a stall is a real risk but not a property of an architecture.

Now split it by the only column that matters: whether one search was ever going to be enough.

In [16]:
by_hop = RESULTS.pivot_table(index="one_search_enough", columns="arm",
                             values=["route", "cited", "complete"], aggfunc="mean")
print(by_hop.round(2).to_string())
print("\ncost of the same six tickets:")
print(RESULTS.groupby("arm")[["tokens", "usd"]].sum().reindex(ORDER).round(3).to_string())

                  cited          complete          route         
arm               agent workflow    agent workflow agent workflow
one_search_enough                                                
False               0.5      0.5      0.5      0.5   1.0     1.00
True                1.0      1.0      1.0      1.0   1.0     0.75

cost of the same six tickets:
          tokens   usd
arm                   
workflow   18335  0.01
agent      64855  0.03


Sit with that for a moment, because it is not what the room expects and it is not what you were sold.

**On answer quality, the rung bought nothing.** Same band on all three checks. The agent did not answer more tickets correctly than the graph did; on the four tickets one search settles it matched, and on the two that need a second lookup it failed the same way the workflow did.

**On cost, the rung charged about three times.** Three to four times the tokens, three times the wall clock, and a bill that scales with every ticket the desk ever raises.

If you stop the lab here, the honest summary is the one S20 predicted and most teams find out later: *the loop was not the missing piece.* Something is genuinely wrong on those two tickets, and climbing a rung did not touch it. The next three sections are about what it actually is.

In [17]:
missing = RESULTS[~RESULTS.one_search_enough]
for _, row in missing.iterrows():
    rows = WF if row.arm == "workflow" else AG
    record = next(r["record"] for r in rows if r["ticket_id"] == row.ticket)
    print(f"{row.arm:<9} {row.ticket}  cited={row.cited}  complete={row.complete}")
    print("   cited:", ", ".join(record["citations"]) or "(nothing)")
    print("   missing:", record["missing"] or "(reported nothing missing)")
    print()

workflow  SD-2026-0435  cited=1.0  complete=False
   cited: MAN-FW-01 rev 3, WO-2026-0266 rev 1
   missing: (reported nothing missing)

workflow  SD-2026-0412  cited=0.0  complete=True
   cited: MAN-GD-01 rev 2, LOG-2026-06-11-N rev 1
   missing: (reported nothing missing)

agent     SD-2026-0435  cited=1.0  complete=False
   cited: MAN-FW-01 rev 3, WO-2026-0266 rev 1
   missing: (reported nothing missing)

agent     SD-2026-0412  cited=0.0  complete=True
   cited: MAN-GD-01 rev 2, LOG-2026-06-11-N rev 1
   missing: (reported nothing missing)



There is the practical finding, and it is worth more than the score column.

A node that is asked *what could you not reach* will sometimes tell you. Where it does, an incomplete answer becomes a ticket a human finishes in thirty seconds. Where it does not — where `missing` is empty and the answer is half of one — you have S20's rung-4 failure exactly: *a bad step-two output carried silently into step five.* The difference between those two systems is one field in a schema and one line in a prompt, and neither of them is a rung.

## 7. The currency you spent

S20 said determinism falls off a cliff between rung 4 and rung 5, and that the cliff is what your auditor asks about. Three measurements, and they are not the same claim.

**How much does the work vary across inputs?**

In [18]:
spread = RESULTS.groupby("arm")[["tool_calls", "model_calls", "tokens"]].agg(["min", "max", "std"])
print(spread.reindex(ORDER).round(1).to_string())

         tool_calls          model_calls          tokens               
                min max  std         min max  std    min    max     std
arm                                                                    
workflow          3   4  0.4           2   2  0.0   2276   3409   402.2
agent             2   3  0.4           4   5  0.4   7066  11992  1903.6


The workflow's spread is a property of the drawing: three tool calls, four when the switch fires, two model calls — every time, for every ticket, including the ticket nobody has written yet. That goes in a capacity plan before you run it once.

The agent's spread is a property of *these six tickets*. It is an observation, not a bound, and next month's tickets are not obliged to respect it.

**How many paths are there, and how many did you see?**

In [19]:
def signature(row: dict) -> str:
    return " -> ".join(c["tool"].replace("docs__", "").replace("desk__", "") for c in row["calls"])


for arm, rows in (("workflow", WF), ("agent", AG)):
    taken = sorted({signature(r) for r in rows})
    print(f"{arm}: {len(taken)} distinct paths actually taken across {len(rows)} tickets")
    for path in taken:
        print("   ", path)
print()

branches = [n.name for n in WORKFLOW.nodes if n.when is not None]
print(f"paths the workflow can take: {2 ** len(branches)}  (one switch, on {', '.join(branches)})")
print(f"paths the agent can take:    up to {len(TOOLS_OFFERED) ** AGENT_STEPS:,}  "
      f"({len(TOOLS_OFFERED)} tools, {AGENT_STEPS} steps)\n")

workflow: 2 distinct paths actually taken across 6 tickets
    get_ticket -> find_similar_tickets -> list_tickets
    get_ticket -> find_similar_tickets -> list_tickets -> search_documents
agent: 2 distinct paths actually taken across 6 tickets
    get_ticket -> find_similar_tickets
    get_ticket -> find_similar_tickets -> search_documents

paths the workflow can take: 2  (one switch, on evidence)
paths the agent can take:    up to 5,764,801  (7 tools, 8 steps)



That pair of numbers is the session in one screen.

Both arms took two paths. One of them could only ever have taken two, and you can name both before breakfast. The other had a space of millions and used two of them, which is not a guarantee about anything — it is a sample, taken on six tickets, on a Wednesday, against one snapshot of one model.

And notice *why* the agent's two paths look so much like the graph: the task it was given is a numbered list. The server's `triage_ticket` prompt is a flowchart written in prose. Handing a flowchart to a loop does not make the flowchart go away; it makes it unenforceable, uninspectable and repriced per ticket.

**Does the same ticket give the same path twice?** Both arms run at temperature 0, which sounds like it settles the matter.

In [20]:
if HAVE_MODEL:
    async with McpTools({"docs": DOCS, "desk": DESK}) as tools, McpTools({"docs": DOCS}) as docs_only:
        repeats = {}
        for arm, fn in (("workflow", workflow_once), ("agent", agent_once)):
            repeats[arm] = [await fn(tools, docs_only, "SD-2026-0412") for _ in range(2)]
    for arm, pair in repeats.items():
        print(f"{arm}: identical path on two runs of the same ticket? "
              f"{signature(pair[0]) == signature(pair[1])}")
        for i, row in enumerate(pair, 1):
            tokens = row["prompt_tokens"] + row["completion_tokens"]
            print(f"   run {i}  {row['steps']} steps  {tokens} tokens  {signature(row)}")
        print()
else:
    print("no model: skipped. The claim below is the one to read anyway.")

workflow: identical path on two runs of the same ticket? True
   run 1  6 steps  3231 tokens  get_ticket -> find_similar_tickets -> list_tickets -> search_documents
   run 2  6 steps  3197 tokens  get_ticket -> find_similar_tickets -> list_tickets -> search_documents

agent: identical path on two runs of the same ticket? True
   run 1  3 steps  11938 tokens  get_ticket -> find_similar_tickets -> search_documents
   run 2  3 steps  11844 tokens  get_ticket -> find_similar_tickets -> search_documents



Read those two runs as a sample, not a verdict.

If the agent repeated itself exactly, it will keep doing so right up until the day it does not — a retrained endpoint, a tie between two tool descriptions, a passage that came back in a different order. If it did not repeat itself, look at the token counts: that is the same ticket, the same prompt, the same temperature, costing what it costs. **That spread is the thing to put in the business case**, and it is why S20 says price rung 5 at its worst row.

And if the workflow's two runs differed, look at *where*. The same layers ran in the same order both times; what moved was a model's answer inside one node, which can move whether the switch fires. **Rung 4 does not buy you one path. It buys you a small number of paths you can name**, and a model call inside a node is still a model call. Anyone who sold rung 4 as "deterministic" full stop was overselling; the distinction survives the oversell, and it is the distinction your incident review needs.

An agent run you cannot replay is an agent run you cannot defend. The workflow is defended by code you already version. The agent is defended by a trace you have to have written first — section 11.

## 8. The two tickets nobody reached

SD-2026-0412 in full, both arms. The caller asks what has to happen before a gas detector that failed calibration goes back in service.

In [21]:
def show(ticket_id: str, rows_by_arm: dict) -> None:
    print(f"=== {ticket_id} ===")
    print("gold:", GOLD_BY_ID[ticket_id]["about"])
    print("must cite:", ", ".join(GOLD_BY_ID[ticket_id]["cite"]) or "nothing", "\n")
    for arm, rows in rows_by_arm.items():
        row = next(r for r in rows if r["ticket_id"] == ticket_id)
        rec, sc = row["record"], score(row["record"], GOLD_BY_ID[ticket_id])
        print(f"--- {arm}  ({row['steps']} steps, {len(row['calls'])} tool calls, "
              f"{usd(row['prompt_tokens'], row['completion_tokens']) * 1000:.2f} USD/1000)  {sc}")
        print("   path:", signature(row))
        print("   cites:", ", ".join(rec["citations"]) or "(nothing)")
        print("   answer:", rec["answer"][:600].replace("\n", "\n            "))
        print()


show("SD-2026-0412", {"workflow": WF, "agent": AG})

=== SD-2026-0412 ===
gold: MAN-GD-01 names the override permit; MAN-SIS-01 is what it means
must cite: MAN-GD-01, MAN-SIS-01 

--- workflow  (6 steps, 4 tool calls, 1.75 USD/1000)  {'route': True, 'cited': 0.0, 'complete': True}
   path: get_ticket -> find_similar_tickets -> list_tickets -> search_documents
   cites: MAN-GD-01 rev 2, LOG-2026-06-11-N rev 1
   answer: The gas detector GD-3107 failed its six-monthly calibration with a span reading about 30% low, so it was inhibited under an override permit and a portable monitor was placed at the location. According to the maintenance manual MAN-GD-01 rev 2, section 3, a detector that fails calibration must have its sensor head replaced before returning to service. The sensor head was replaced and a bump test was performed with 25 ppm H2S, reaching the high alarm, after which the inhibit was removed and the override register updated (LOG-2026-06-11-N rev 1). The sensor life section (MAN-GD-01 rev 2, section 

--- agent  (3 steps, 3 tool 

Both arms found MAN-GD-01, which is the right document, and both are correct about the sensor head and the bump test. Neither of them reads the other half.

MAN-GD-01 says the detector "is inhibited under an override permit". It does not say what an override permit requires, because that lives in MAN-SIS-01: Area Authority approval before it goes on, compensating measures written into the permit, an entry in the override register, Plant Manager approval and a written risk assessment past twelve hours, and a management of change past seventy-two. The detector was swapped on the 28th and the desk's clock reads the 29th, so the first threshold has already passed and the second is about a day out. An answer that stops at the bump test is not slightly incomplete. It is missing the half that has a permit, an approver and a clock on it.

SD-2026-0435 is the same shape: the permitted vendor flow is MAN-FW-01 section 3, and the approval that flow needs — change advisory board, or the OT Lead in an emergency with retrospective CAB review — is section 2 of the same document, which the search never returned.

So: one architecture searched once because that is all it was built to do, and the other searched once because it decided that was enough. **Rung 5 does not buy you the second lookup. It buys you the possibility of one, and on this set it did not take it.**

Which raises the question S20's table actually poses, and it is not "workflow or agent". It is: *can you list the case?*

## 9. The third node, before you reach for the loop

Read S20's threshold row again, with the emphasis where it was written:

> Handle a request where step three depends on what step two found, **and you cannot list the cases** → rung 5.

Can we list this case? Here it is, in one sentence: *if the passages point at a document or a section you have not read, read that one too.* That is one case. It fits on a line. It is an `if`.

So it is not a rung-5 requirement at all. It is a missing edge in a rung-4 graph, and the fix is two more nodes: one constrained call that reads the passages and names what is missing, and one more search when there is something to search for. Three model calls instead of two, one more possible tool call, four paths instead of two — and every one of the four still drawable, testable and priceable in advance.

In [22]:
FOLLOWUP_SCHEMA = {
    "type": "object",
    "properties": {
        "need_more": {"type": "boolean",
                      "description": "True only if the passages point at something unread that the answer needs."},
        "query": {"type": "string", "description": "The second search query. Empty when need_more is false."},
        "what_is_missing": {"type": "string"},
    },
    "required": ["need_more", "query", "what_is_missing"],
    "additionalProperties": False,
}


async def n_followup(s):
    prompt = f"""{HOUSE_RULES}

{COMPLETENESS}

TICKET
{brief(s['ticket'])}

WHAT THE FIRST SEARCH RETURNED
{passages(s['evidence'])}

You get one more search, and only one. Do these passages name a document, a procedure or a section
you have not read that the answer depends on — a permit, a standard, an approval, a second code?
If they do, write the query that would find it. If the answer is already whole, say so and do not
search: a search you do not need costs the same as one you do."""
    return await ask_json_async(prompt, FOLLOWUP_SCHEMA, "followup")


async def n_evidence2(s):
    return await call(s, "docs__search_documents", {"query": s["followup"]["query"], "k": 5})


async def n_proposal2(s):
    first = passages(s["evidence"]) if s["evidence"] is not None else "(no search was run)"
    second = passages(s["evidence2"]) if s["evidence2"] is not None else "(no second search was needed)"
    prompt = proposal_prompt(s["ticket"], s["triage"]["route"],
                             f"FIRST SEARCH\n{first}\n\nSECOND SEARCH\n{second}")
    return await ask_json_async(prompt, PROPOSAL_SCHEMA, "proposal")


searched = lambda s: s["evidence"] is not None
needs_more = lambda s: s["followup"] is not None and s["followup"]["need_more"] and s["followup"]["query"]

TWO_HOP = Graph("two-hop-triage", [
    Node("ticket",    (),                              n_ticket,    "tool",  "desk__get_ticket"),
    Node("similar",   ("ticket",),                     n_similar,   "tool",  "desk__find_similar_tickets"),
    Node("queue",     ("ticket",),                     n_queue,     "tool",  "desk__list_tickets"),
    Node("triage",    ("ticket", "similar", "queue"),  n_triage,    "model", "route + the first query"),
    Node("evidence",  ("triage",),                     n_evidence,  "tool",  "docs__search_documents",
         when=lambda s: s["triage"]["route"] != "route_elsewhere",
         when_note="skipped when the ticket is not ours to answer"),
    Node("followup",  ("ticket", "evidence"),          n_followup,  "model", "what did the passages point at?",
         when=searched, when_note="only if a search ran"),
    Node("evidence2", ("followup",),                   n_evidence2, "tool",  "docs__search_documents, once more",
         when=needs_more, when_note="only if something unread was named"),
    Node("proposal",  ("ticket", "triage", "evidence", "evidence2"), n_proposal2, "model",
         "the record the desk would write"),
])
print(TWO_HOP.draw())

two-hop-triage: 8 nodes in 7 layers, 3 model calls, up to 5 tool calls

  1  ticket       tool   desk__get_ticket
  2  similar      tool   desk__find_similar_tickets  <- this layer runs together
  2  queue        tool   desk__list_tickets
  3  triage       model  route + the first query
  4  evidence     tool   docs__search_documents  [skipped when the ticket is not ours to answer]
  5  followup     model  what did the passages point at?  [only if a search ran]
  6  evidence2    tool   docs__search_documents, once more  [only if something unread was named]
  7  proposal     model  the record the desk would write


In [23]:
async def two_hop_once(tools, docs_only, ticket_id: str, verbose: bool = False) -> dict:
    METER.take()
    start = time.time()
    state, log = await TWO_HOP.run(ticket_id=ticket_id, tools=tools, calls=[])
    return {"arm": "two-hop", "ticket_id": ticket_id, "record": state["proposal"],
            "triage": state["triage"], "followup": state["followup"], "calls": state["calls"],
            "log": log, "steps": sum(r["ran"] for r in log), "capped": False,
            "seconds": round(time.time() - start, 2), **METER.take()}


W2 = await run_arm("two-hop", two_hop_once)

two-hop: 6 runs loaded from outputs/12_agent_graph/runs/two-hop.json (set FORCE=True to re-run)


In [24]:
for row in W2:
    followup = row.get("followup") or {}
    print(f"{row['ticket_id']}  second search: {followup.get('need_more')}")
    print(f"   {followup.get('what_is_missing', '')[:150]}")
    if followup.get("query"):
        print(f"   query: {followup['query']}")

SD-2026-0409  second search: False
   The passages provide the error code meaning, the action to take, and the history of the issue including the restart consequences and archive volume st
SD-2026-0427  second search: False
   The passages provide the error codes HX-4417 and HX-4471 meanings and actions from MAN-HIS-01 rev 5, and the context of the issue from WO-2026-0201 re
SD-2026-0421  second search: False
   The passages provide the maximum discharge pressure for pumps, but none mention P-301 specifically. The documents found are for P-202, K-301, and P-10
SD-2026-0423  second search: None
   
SD-2026-0435  second search: True
   The passages mention that remote vendor support is allowed from the DMZ jump host to EWS-01 when a permit is active, but do not specify the procedure 
   query: permit procedure for remote vendor support on FW-OT-01
SD-2026-0412  second search: False
   The passages provide a complete procedure for handling the failed calibration of GD-3107, including inhi

That middle column is the lab's argument in one place. The node was handed passages rather than a ticket, and asked a question it could only answer because it had read something first. **That is "step three depends on what step two found", and it is an `if` statement.**

Now look at how often it fired, and on which ticket it did not.

In [25]:
show("SD-2026-0412", {"workflow": WF, "two-hop": W2, "agent": AG})

=== SD-2026-0412 ===
gold: MAN-GD-01 names the override permit; MAN-SIS-01 is what it means
must cite: MAN-GD-01, MAN-SIS-01 

--- workflow  (6 steps, 4 tool calls, 1.75 USD/1000)  {'route': True, 'cited': 0.0, 'complete': True}
   path: get_ticket -> find_similar_tickets -> list_tickets -> search_documents
   cites: MAN-GD-01 rev 2, LOG-2026-06-11-N rev 1
   answer: The gas detector GD-3107 failed its six-monthly calibration with a span reading about 30% low, so it was inhibited under an override permit and a portable monitor was placed at the location. According to the maintenance manual MAN-GD-01 rev 2, section 3, a detector that fails calibration must have its sensor head replaced before returning to service. The sensor head was replaced and a bump test was performed with 25 ppm H2S, reaching the high alarm, after which the inhibit was removed and the override register updated (LOG-2026-06-11-N rev 1). The sensor life section (MAN-GD-01 rev 2, section 

--- two-hop  (7 steps, 4 too

It fired once in six, and not on the ticket this section was built for.

On SD-2026-0412 the follow-up node read a passage saying the detector "is inhibited under an override permit", decided the answer was whole, and did not search. It is not being lazy. It cannot know that *override permit* is a term with three pages of its own somewhere in the store, because nothing in its context says what the store contains.

So the third node did not reach it either. Four architectures now — two paths, eight paths, a capped loop, an open loop — and the same two tickets come back half-answered from every one of them. That is the point to stop climbing and ask what the failure actually is.

In [26]:
# No model in this cell. One search, with a query written by a human who knows the corpus.
async with McpTools({"docs": DOCS}) as docs_only:
    hits = as_json(await docs_only.call(
        "docs__search_documents",
        {"query": "override permit approval requirements for an inhibited safety function", "k": 3}))
for hit in (hits if isinstance(hits, list) else [hits]):
    print(f"{hit['doc_id']} rev {hit['revision']}  [{hit['score']:.1f}]  {hit['section']}")
    print("   ", hit["text"][:220].replace("\n", " "))

MAN-SIS-01 rev 2  [18.7]  2. Approval
    Safety Instrumented System and F&G Override Management [MAN-SIS-01 rev 2, current] Section: 2. Approval  - Every override needs an override permit approved by the Area Authority before it is applied. - An override longer
WO-2026-0281 rev 1  [15.9]  Work performed
    Work Order WO-2026-0281 - PT-3105 compressor suction pressure transmitter [WO-2026-0281 rev 1, current] Section: Work performed  Trip function on PT-3105 overridden under an override permit approved by the Area Authority
MAN-SIS-01 rev 2  [15.7]  1. Principle
    Safety Instrumented System and F&G Override Management [MAN-SIS-01 rev 2, current] Section: 1. Principle  An override (also called a bypass or inhibit) of a safety instrumented function or of a fire and gas detector remo


One search. Top hit, MAN-SIS-01, the approval rules for the override permit. It was one query away for the whole lab, and no amount of control flow found it.

**That is not an agent problem. It is a retrieval problem wearing an agent costume** — and it is the most expensive mistake in this field. The system is a bit wrong, the diagnosis jumps to autonomy, and a team spends a quarter building a loop that re-asks the question it was always going to ask. S19 settled the tune-versus-retrieve version of this argument on Day 4 morning. This is the same argument, one rung up, and it costs more to get wrong.

Section 13 has the two fixes that would actually work on those two tickets. Neither of them is a rung.

## 10. And when you genuinely cannot list the case

The third node works because the case was listable. Some are not: a caller who needs four unrelated things, a fault whose next lookup depends on a number in the last one, an investigation that is finished when it is finished. For those, the shape is not a bigger graph and it is not an open loop either. It is **the graph, with the loop confined to the one node you could not draw**, under two controls:

| Control | Value here | Why it is not in the prompt |
|---|---|---|
| step cap | 6 | a cap in a prompt is a suggestion; this one is a `for` loop that ends |
| tool list | the document server only | it cannot touch a ticket because it was never told tickets exist |

The second row is S22's takeaway doing real work. The desk tools are not gated inside this node, they are *absent*. A capability you have to remember not to use is not a control.

In [27]:
HYBRID_STEPS = 6  # the cap. Four was the first guess and it bit; see section 13 before you argue for it


async def n_investigate(s):
    """The one node nobody can draw: look things up until the answer is whole, or six steps."""
    ticket = s["ticket"]
    task = (f"Ticket {ticket['ticket_id']} on {ticket['affected_system']}: {ticket['summary']}\n\n"
            f"Desk note: {ticket.get('latest_note') or '(none)'}\n\n"
            f"Find everything in the plant document store the desk needs to answer this. "
            f"{COMPLETENESS} Quote the document id, revision and section for each fact. If the store "
            "does not cover the system or the code the ticket names, say so plainly and stop.")
    result = await run_agent(s["docs_tools"], task, client=METER, model=MODEL, gate=allow_all,
                             system=HOUSE_RULES, max_steps=HYBRID_STEPS, verbose=False)
    for step in result["trace"]:
        s["calls"].append({"tool": step["tool"], "args": step["args"], "chosen_by": "model",
                           "seconds": None})
    return result


async def n_proposal_hybrid(s):
    return await ask_json_async(
        proposal_prompt(s["ticket"], "", s["investigate"]["answer"]), PROPOSAL_SCHEMA, "proposal")


HYBRID = Graph("hybrid-triage", [
    Node("ticket",      (),           n_ticket,      "tool",  "desk__get_ticket"),
    Node("similar",     ("ticket",),  n_similar,     "tool",  "desk__find_similar_tickets"),
    Node("queue",       ("ticket",),  n_queue,       "tool",  "desk__list_tickets"),
    Node("investigate", ("ticket",),  n_investigate, "agent",
         f"docs server only, {HYBRID_STEPS} steps max"),
    Node("proposal",    ("ticket", "similar", "queue", "investigate"), n_proposal_hybrid, "model",
         "the record the desk would write"),
])
print(HYBRID.draw())

hybrid-triage: 5 nodes in 3 layers, 1 model calls, up to 3 tool calls, 1 capped loop

  1  ticket       tool   desk__get_ticket
  2  similar      tool   desk__find_similar_tickets  <- this layer runs together
  2  queue        tool   desk__list_tickets
  2  investigate  agent  docs server only, 6 steps max
  3  proposal     model  the record the desk would write


Note which layer `investigate` landed in. It needs only the ticket, so it starts at the same moment as the two desk reads rather than after them: the expensive node begins first, because you were the one scheduling. A loop cannot do that to itself.

In [28]:
async def hybrid_once(tools, docs_only, ticket_id: str, verbose: bool = False) -> dict:
    METER.take()
    start = time.time()
    state, log = await HYBRID.run(ticket_id=ticket_id, tools=tools, docs_tools=docs_only, calls=[])
    return {"arm": "hybrid", "ticket_id": ticket_id, "record": state["proposal"],
            "calls": state["calls"], "log": log, "trace": state["investigate"]["trace"],
            "steps": sum(r["ran"] for r in log) + state["investigate"]["steps"],
            "capped": state["investigate"]["capped"],
            "seconds": round(time.time() - start, 2), **METER.take()}


HY = await run_arm("hybrid", hybrid_once)
RESULTS = pd.concat([rows_to_frame(rows) for rows in (WF, W2, HY, AG)], ignore_index=True)
ORDER = ["workflow", "two-hop", "hybrid", "agent"]
RESULTS.groupby("arm").agg(
    route=("route", "mean"), cited=("cited", "mean"), complete=("complete", "mean"),
    tool_calls=("tool_calls", "mean"), model_calls=("model_calls", "mean"),
    tokens=("tokens", "mean"), usd_per_1000=("usd", lambda c: c.mean() * 1000),
    seconds=("seconds", "median"), capped=("capped", "sum"),
).reindex(ORDER).round(2)

hybrid: 6 runs loaded from outputs/12_agent_graph/runs/hybrid.json (set FORCE=True to re-run)


,route,cited,complete,tool_calls,model_calls,tokens,usd_per_1000,seconds,capped
arm,,,,,,,,,
workflow,0.83,0.75,0.83,3.83,2.00,3055.83,1.60,5.74,0
two-hop,0.83,0.75,0.83,4.00,2.83,4331.83,2.18,6.94,0
hybrid,0.83,0.50,0.83,6.17,4.67,8450.83,4.04,15.76,1
agent,1.00,0.75,0.83,2.83,4.83,10809.17,5.08,14.99,0


In [29]:
print("citations complete, split by whether one search was ever going to be enough:")
print(RESULTS.pivot_table(index="one_search_enough", columns="arm", values="cited",
                          aggfunc="mean")[ORDER].round(2).to_string())
print("\nwhere the next step came from:")
origin = pd.DataFrame([
    {"arm": r["arm"], "by code": sum(c["chosen_by"] == "code" for c in r["calls"]),
     "by model": sum(c["chosen_by"] == "model" for c in r["calls"])}
    for rows in (WF, W2, HY, AG) for r in rows
]).groupby("arm").mean().reindex(ORDER).round(2)
print(origin.to_string())
print("\npaths, by arm:")
for name, graph in (("workflow", WORKFLOW), ("two-hop", TWO_HOP), ("hybrid", HYBRID)):
    switches = sum(1 for n in graph.nodes if n.when is not None)
    loops = sum(1 for n in graph.nodes if n.kind == "agent")
    bound = (f"{2 ** switches} × up to {len(TOOLS_OFFERED) ** HYBRID_STEPS:,} inside one node"
             if loops else f"{2 ** switches}")
    print(f"  {name:<9} {bound}")
print(f"  {'agent':<9} up to {len(TOOLS_OFFERED) ** AGENT_STEPS:,}")

citations complete, split by whether one search was ever going to be enough:
arm                workflow  two-hop  hybrid  agent
one_search_enough                                  
False                   0.5      0.5     0.5    0.5
True                    1.0      1.0     0.5    1.0

where the next step came from:
          by code  by model
arm                        
workflow     3.83      0.00
two-hop      4.00      0.00
hybrid       3.00      3.17
agent        0.00      2.83

paths, by arm:
  workflow  2
  two-hop   8
  hybrid    1 × up to 117,649 inside one node
  agent     up to 5,764,801


That last block is the one to photograph, because it is the same measurement S20's whole session was about, taken on your own system: **what fraction of the next steps is decided by code somebody can read, and how many paths does that leave to test.**

Read the four rows as an argument settled by measurement rather than by preference.

- **On answer quality, no arm was reliably better.** Routes and completeness land in the same band across a ladder running from two paths to five and a half million, and where the citation column does move it is the one with the loop in it that is behind.
- **On cost, three times separated them**, charged per ticket, forever.
- **On what you can test, everything separated them.** Two paths you can enumerate over coffee, against a space nobody will ever sample.

So the recommendation for this job is the boring one, and it is the one the numbers make rather than the one the room expected: **ship the workflow.** Climb when you have a failure that is genuinely about control flow — and check first, as section 9 did in a single cell, that it is not something cheaper in a costume.

The hybrid remains the shape to reach for when the case truly cannot be listed, and S24 hangs its controls on exactly that node. Look at its `capped` column before you decide: where the cap bit, the node returned an honest partial instead of a confident whole. That is what a cap buys, and what it costs.

Nobody should leave this room able to say "we need an agent" without naming which of those four rows they are on, what the row above could not reach, and what they measured.

## 11. What you hand to whoever asks on Monday

Everything above is a decision somebody will eventually have to defend: a ticket that got the wrong priority, an answer that quoted a withdrawn revision, a run that cost forty times the median. For rung 4 the defence is the code, and the code is in version control. For rung 5 the defence is the trace, and a trace printed to a cell and scrolled past does not exist.

In [30]:
scores_path = OUT / "scores.jsonl"          # contract 4: one row per arm per ticket
RESULTS.to_json(scores_path, orient="records", lines=True)

for rows in (WF, W2, HY, AG):
    for row in rows:
        (OUT / "traces" / f"{row['arm']}_{row['ticket_id']}.json").write_text(
            json.dumps(row, indent=2, ensure_ascii=False), encoding="utf-8")

errors = [(r["arm"], r["ticket_id"], step["tool"], (step["output"] or "").replace("\n", " ")[:100])
          for rows in (AG, HY) for r in rows for step in r.get("trace", [])
          if "TOOL ERROR" in (step["output"] or "")]
print("tool calls the servers refused or could not serve:", len(errors))
for row in errors[:8]:
    print("  ", " | ".join(row))

written = sorted(p.relative_to(ROOT) for p in OUT.rglob("*") if p.is_file())
print(f"\n{len(written)} files under {OUT.relative_to(ROOT)}; one trace, as an auditor reads it:")
sample = next(r for r in AG if r["ticket_id"] == "SD-2026-0412")
for step in sample["trace"]:
    print(f"  step {step['step']}  {'ok    ' if step['allowed'] else 'DENIED'}  {step['tool']}"
          f"({json.dumps(step['args'])[:80]})")

tool calls the servers refused or could not serve: 6
   hybrid | SD-2026-0409 | docs__get_document | TOOL ERROR: Error executing tool get_document: MAN-HIS-01 has no revision 5. Available: ['MAN-HIS-01
   hybrid | SD-2026-0409 | docs__get_document | TOOL ERROR: Error executing tool get_document: RCA-2026-003 has no revision 1. Available: ['RCA-2026
   hybrid | SD-2026-0409 | docs__get_document | TOOL ERROR: Error executing tool get_document: LOG-2026-04-22-D has no revision 1. Available: ['LOG-
   hybrid | SD-2026-0423 | docs__get_document | TOOL ERROR: Error executing tool get_document: PLAN-2026 has no revision 1. Available: ['PLAN-2026']
   hybrid | SD-2026-0435 | docs__get_document | TOOL ERROR: Error executing tool get_document: MAN-FW-01 has no revision 3. Available: ['MAN-FW-01']
   hybrid | SD-2026-0435 | docs__get_document | TOOL ERROR: Error executing tool get_document: MAN-HMI-01 has no revision 2. Available: ['MAN-HMI-01

32 files under outputs/12_agent_graph; one trace, as

Two things in that output rather than one.

**The trace is the bill for crossing the line.** Tool, arguments, whether the gate allowed it, what came back, in order, per run, kept as long as your retention policy says. Nobody writes that into a pilot, and every pilot that reaches production has it retrofitted by somebody who was not in the room when the decisions were made.

**The errors are part of the interface, and they are a design conversation.** Most of those are the same shape: a model asking for `revision: 5` of a document that has only ever had one revision, and a server answering with what exists. Is that the model's mistake or the tool's contract? You can only have that argument because the trace recorded both sides of it — and the loops recovered, because the refusal was written to be read. S22 made that point with a gate; here it is with a typo.

## 12. Your rung, in three lines

Before the break, S20 asked each group for three lines. You now have numbers to write them against, from your own run rather than from a slide. Fill these in for your capstone, not for this ticket queue.

1. **The rung our capstone needs**, as a number.
2. **The failure of the rung below that justifies it** — one sentence, concrete, naming the thing rung *n−1* could not reach.
3. **The thing that would make us climb down a rung.**

The third line is still the one that matters. And after this lab there is a fourth question that comes before all of them: *is the failure we named actually a control-flow problem?* Twice today the answer was no — it was one more edge, and one more `if`.

In [31]:
summary = RESULTS.groupby("arm").agg(
    scored=("route", "size"), route=("route", "mean"), cited=("cited", "mean"),
    complete=("complete", "mean"), usd_per_1000=("usd", lambda c: c.mean() * 1000),
    tool_calls=("tool_calls", "mean"), seconds=("seconds", "median")).reindex(ORDER).round(2)

worksheet = OUT / "rung_worksheet.md"
worksheet.write_text(f"""# Which rung does our capstone need?

Measured in lab 12 on {len(TICKET_IDS)} SGP service desk tickets, model `{MODEL}`, the same two MCP
servers, the same house rules and the same output record for every arm. `usd_per_1000` is the cost
of a thousand tickets.

{summary.to_markdown()}

Citations complete, split by whether one document search was ever going to be enough:

{RESULTS.pivot_table(index="one_search_enough", columns="arm", values="cited", aggfunc="mean")[ORDER].round(2).to_markdown()}

Next steps decided by code you can read, versus by a model at runtime:

{origin.to_markdown()}

## Our three lines

1. The rung our capstone needs:
2. The failure of the rung below that justifies it (concrete, name what it could not reach):
3. What would make us climb back down a rung:

## Before anyone builds rung 5

- [ ] The failure we are climbing for is a control-flow failure, not a retrieval or a prompt one
- [ ] One more node or one more `if` has been tried first, and did not fix it
- [ ] Step cap, money cap, wall-clock cap — written down, not intended
- [ ] The trace is stored, and somebody has actually read one
- [ ] An end-to-end eval set with known-good outcomes, including cases that should abstain
- [ ] The write, if there is one, is gated separately from the autonomy (S24)
- [ ] Somebody can say what the system costs at its worst row, not its average
""", encoding="utf-8")
print(worksheet.read_text(encoding="utf-8")[:1400])

# Which rung does our capstone need?

Measured in lab 12 on 6 SGP service desk tickets, model `gpt-4.1-mini`, the same two MCP
servers, the same house rules and the same output record for every arm. `usd_per_1000` is the cost
of a thousand tickets.

| arm      |   scored |   route |   cited |   complete |   usd_per_1000 |   tool_calls |   seconds |
|:---------|---------:|--------:|--------:|-----------:|---------------:|-------------:|----------:|
| workflow |        6 |    0.83 |    0.75 |       0.83 |           1.6  |         3.83 |      5.74 |
| two-hop  |        6 |    0.83 |    0.75 |       0.83 |           2.18 |         4    |      6.94 |
| hybrid   |        6 |    0.83 |    0.5  |       0.83 |           4.04 |         6.17 |     15.76 |
| agent    |        6 |    1    |    0.75 |       0.83 |           5.08 |         2.83 |     14.99 |

Citations complete, split by whether one document search was ever going to be enough:

| one_search_enough   |   workflow |   two-hop |   hybri

## 13. Try it, if the group is ahead

Five changes, each one cell, each measurable against the table you already have.

**Move the switch.** The workflow classifies before it searches, which is why a ticket that reads like its neighbour gets routed `duplicate` before anything has been read. Move `triage` after `evidence` — search first, classify on the passages — and re-run with `FORCE = True`. A mis-route fixed by reordering a graph is a rung-4 fix to a rung-4 problem, which is the whole argument of this lab in one edit.

**Tighten the cap until it bites.** Set `HYBRID_STEPS = 4` and re-run the hybrid. One ticket will come back `no_document` where it previously answered: the node spends two of its four steps on a document id it gets slightly wrong, and runs out. Note *how* it failed — honestly, with a partial — and decide whether you would rather have had a guess.

**Break a node and watch nobody notice.** Make `n_triage` always return `documents`. Every ticket still gets an answer, and the budget question now gets a confident document one. That is S20's rung-4 failure mode, and it is why unit tests on each node tell you nothing about the system.

**Raise the agent's cap and look for the tail.** `AGENT_STEPS = 20`, re-run the agent arm, and watch the token column. Then decide what you would have set the cap to if you were paying per run at a thousand tickets a month.

**Tell it what exists.** `docs__list_documents` was on the table all afternoon and no arm called it. Put the document id list into the follow-up node's prompt — it is about forty ids — and re-run the two-hop arm. A node that can see MAN-SIS-01 in a list has a chance of asking for it; one that cannot is guessing at a corpus it has never been shown.

**Swap the retriever.** Set `SGP_DOCS_RETRIEVAL=hybrid` in the `DOCS` environment and re-run. Lab 07's dense-plus-BM25-plus-rerank pipeline is now behind an identical tool contract and nothing else in this notebook changes. Does the *first* search reach MAN-SIS-01? If it does, you have bought with a retriever what four architectures could not buy with control flow — the cheapest trade in this lab, and the one nobody tries first.

## What to take away

- **The line is who decides the next step, and it is a column in your own table.** `by code` versus `by model`, measured, per arm. Not a philosophy.
- **On this job the rung was not the variable.** Four architectures, two paths to five and a half million, the same answers, three times the cost. Build two and measure: it costs an afternoon and settles the argument for a year.
- **Rung 5 does not buy you a second lookup. It buys the possibility of one**, and on this set it did not take it. That is not a foundation for a promise to a caller.
- **Check whether your agent problem is a retrieval problem in costume.** The document that defeated all four arms was one search away from a human who knew the corpus. Diagnose the failure before you choose the architecture; that is S19's discipline, one rung up.
- **A node can only ask for what it can imagine.** The two-hop graph is still the right first move and costs a third of the loop, but "what am I missing" is not answerable without knowing what exists. Give it the list before you give it autonomy.
- **A prompt that says "work in this order" is a flowchart.** Handing it to a loop does not remove it; it makes it unenforceable, uninspectable and repriced on every ticket.
- **Determinism is not something you test for.** Rung 4 buys a small number of paths you can name. Nothing buys you one path, because a model call inside a node is still a model call.
- **Put the autonomy where the uncertainty is, cap it, and give it only the tools it needs.** The absent tool is the control. The gated tool is the compromise.
- **Crossing the line makes the trace part of the system.** Rung 4 is defended with code you already version. Rung 5 is defended with a log you had to write first.

## Handoff to S24

You now have four architectures, a measured reason to prefer the cheap one, and a named condition for climbing. Every control in this lab was a default nobody chose: an eight-step cap because the helper shipped with one, a gate that refuses every write because S22 left it that way, one model, no budget, no approval, no retention policy, and an audit file that exists because a notebook cell happened to write it.

S24 is where those stop being defaults:

> It is Wednesday afternoon, the loop is running against the real service desk, and it wants to close a ticket. What stops it, who says yes, what has it cost before anyone notices, and what do you show the auditor in March?

`13_agent_control` starts there, on the graph you just built.

## Facilitator: save this run as the room's fallback

In [32]:
PROMOTE = False  # facilitator only: after a good live run, keep it for when the network or a model fails
if PROMOTE and HAVE_MODEL:
    import shutil
    (PREBAKED / "runs").mkdir(parents=True, exist_ok=True)
    for path in RUNS.glob("*.json"):
        shutil.copy2(path, PREBAKED / "runs" / path.name)
    print("copied", RUNS.relative_to(ROOT), "->", (PREBAKED / "runs").relative_to(ROOT))

# Reset the ticket store so the next person starts from the same twelve tickets. Nothing in this
# lab writes to it — every arm was gated to propose only — but a clean slate is free.
store = OUT / "tickets.json"
if store.exists():
    store.unlink()
    print("ticket store reset")

ticket store reset
